In [108]:
import sys
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd()
if (cwd / "src").exists():
    project_root = cwd
elif (cwd / "integration.py").exists():
    project_root = cwd.parent
else:
    project_root = cwd.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
print("Project root:", project_root)

Project root: c:\Users\Acer\Documents\RAG-from-scratch\RAG-from-scratch


In [109]:
from dotenv import load_dotenv

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage


from langchain_ollama import ChatOllama, OllamaEmbeddings


from data_source.wikipedia import WikipediaSource
from data_source.web_search import WebSearchSource


from src.pipeline.indexing_rag import build_vectorstore
from src.pipeline.retrieval_rag import create_retriever


from src.query_translation.multi_query import (
    create_multi_query_retrieval_chain,
)

from src.Reranking.reranking import CrossEncoderReranker


from src.advanced_indexing.raptor import RaptorIndexer


from src.advanced_RAG.Self_RAG.self_rag import SelfRAG
from src.advanced_RAG.Long_Context.long_context import LongContext


from src.memory.memory import ConversationMemory


from src.evaluation.rag_evaluation import RAGEvaluator

In [110]:
# Load environment variables

load_dotenv()


# Local LLM

llm = ChatOllama(
    model="llama3:latest",
    temperature=0,
    base_url="http://127.0.0.1:11434",
)


# Local embedding model

embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest",
    base_url="http://127.0.0.1:11434",
)

In [111]:
# Final generation prompt

generation_prompt = ChatPromptTemplate.from_template("""
You are a careful RAG assistant.

Answer the question using ONLY the provided evidence.

Evidence:
{context}

Question:
{question}

Rules:
- Do not invent facts.
- Use only the provided evidence.
- For current or time-sensitive questions, prefer the newest
  evidence with an explicit date and time.
- If sources conflict, prefer the most recent dated evidence.
- Do not use older information when newer evidence is available.
- If the evidence is insufficient, say that the information
  is not available in the retrieved evidence.
- Answer clearly and concisely.

Answer:
""")

generation_chain = generation_prompt | llm | StrOutputParser()

In [112]:
# Data sources

wikipedia = WikipediaSource(
    top_k=5,
)

web_search = WebSearchSource(
    top_k=5,
)


# Reranker

reranker = CrossEncoderReranker(
    top_k=5,
)


# Conversation memory

conversation_memory = ConversationMemory()


# Evaluator

evaluator = RAGEvaluator(
    model="llama3:latest",
    temperature=0,
)

In [113]:
# Ollama answerability check

answerability_prompt = ChatPromptTemplate.from_template("""
Determine whether you can answer the user's question reliably
using your existing knowledge.

Return ONLY one of these labels:

ANSWERABLE
NOT_ANSWERABLE

Rules:
- Return NOT_ANSWERABLE for current, live, latest, recent,
  today's, yesterday's, tomorrow's, or time-sensitive information.
- Return NOT_ANSWERABLE if you are uncertain.
- Return ANSWERABLE only when you are reasonably confident
  that your existing knowledge is sufficient.

Question:
{question}

Decision:
""")

answerability_chain = answerability_prompt | llm | StrOutputParser()

In [114]:
# User query

query = input("Ask a question: ")


# Ask Ollama first

decision = (
    answerability_chain.invoke(
        {
            "question": query,
        }
    )
    .strip()
    .upper()
)

if "NOT_ANSWERABLE" in decision:
    decision = "NOT_ANSWERABLE"

elif "ANSWERABLE" in decision:
    decision = "ANSWERABLE"

else:
    # Conservative fallback
    decision = "NOT_ANSWERABLE"

print("Ollama decision:", decision)

Ollama decision: ANSWERABLE


In [115]:
# Determine retrieval path

if decision == "ANSWERABLE":

    response = llm.invoke(query)
    answer = response.content

    print("Answered directly by Ollama.")

else:

    print("Ollama could not answer reliably.")
    print("External retrieval required.")

Answered directly by Ollama.


In [116]:
# Detect current / time-sensitive queries


def is_current_query(query: str) -> bool:
    keywords = [
        "today",
        "current",
        "latest",
        "now",
        "live",
        "recent",
        "yesterday",
        "tomorrow",
    ]

    query_lower = query.lower()

    return any(keyword in query_lower for keyword in keywords)


current_query = is_current_query(query)

print("Current query:", current_query)

Current query: False


In [117]:
# External retrieval

documents = []

if decision == "NOT_ANSWERABLE":

    if current_query:

        # Current / time-sensitive information
        documents = web_search.retrieve(query)

        print("Source: Tavily Web Search")

        print(
            "Documents:",
            len(documents),
        )

    else:

        # Stable external information
        wikipedia_documents = wikipedia.retrieve(query)
        web_documents = web_search.retrieve(query)

        documents = wikipedia_documents + web_documents

        print(
            "Wikipedia:",
            len(wikipedia_documents),
        )

        print(
            "Web:",
            len(web_documents),
        )

        print(
            "Total:",
            len(documents),
        )

In [118]:
# Deduplicate external documents

if decision == "NOT_ANSWERABLE":

    unique_documents = {}

    for document in documents:

        url = document.metadata.get("url")

        if url:
            key = url
        else:
            key = document.metadata.get("title", "") + document.page_content[:200]

        if key not in unique_documents:
            unique_documents[key] = document

    documents = list(unique_documents.values())

    print(
        "Unique documents:",
        len(documents),
    )

In [119]:
for i, document in enumerate(documents, 1):
    print(
        i,
        document.metadata.get("title"),
        document.metadata.get("url"),
    )

In [120]:
# Reranking
if decision == "NOT_ANSWERABLE":
    reranked_documents = reranker.rerank_documents(
        query=query,
        documents=documents,
    )

In [121]:
from datetime import datetime


def extract_document_datetime(document):
    """Extract document datetime from metadata."""
    metadata = getattr(document, "metadata", {}) or {}

    # Try common metadata keys
    for key in ["date", "datetime", "created_at", "published_at", "timestamp"]:
        value = metadata.get(key)

        if value is not None:
            return value

    return None

In [122]:
# Inspect ranking

if decision == "NOT_ANSWERABLE":

    for i, document in enumerate(
        reranked_documents,
        1,
    ):
        print(f"\n--- Rank {i} ---")

        print(
            "Title:",
            document.metadata.get("title"),
        )

        print(
            "URL:",
            document.metadata.get("url"),
        )

        print(
            "Date:",
            extract_document_datetime(document),
        )

        print(document.page_content[:700])

In [123]:
if decision == "NOT_ANSWERABLE" and documents:
    vectorstore = build_vectorstore(
        documents=documents,
        embedding_model=embeddings,
        batch_size=32,
    )

In [124]:
# Build vector store

vectorstore = None
retriever = None
multi_query_retriever = None

if decision == "NOT_ANSWERABLE":

    vectorstore = build_vectorstore(
        documents=documents,
        embedding_model=embeddings,
        batch_size=32,
    )

    print("Vector store created.")

In [125]:
# Create retriever

if decision == "NOT_ANSWERABLE":

    retriever = create_retriever(
        vectorstore=vectorstore,
        k=5,
    )

In [126]:
# Multi-Query retrieval

if decision == "NOT_ANSWERABLE":

    multi_query_retriever = create_multi_query_retrieval_chain(
        retriever=retriever,
        llm=llm,
    )

    retrieved_documents = multi_query_retriever.invoke(query)

    print(
        "Retrieved documents:",
        len(retrieved_documents),
    )

In [127]:
# Rerank stable RAG results

if decision == "NOT_ANSWERABLE":

    reranked_documents = reranker.rerank_documents(
        query=query,
        documents=retrieved_documents,
    )

    print(
        "Stable RAG reranked documents:",
        len(reranked_documents),
    )

In [128]:
# RAPTOR

raptor_leaf = []
raptor_clusters = []

if decision == "NOT_ANSWERABLE":

    raptor = RaptorIndexer(
        llm=llm,
        embeddings=embeddings,
        n_clusters=3,
    )

    raptor.build_tree(
        documents=documents,
    )

    raptor_results = raptor.retrieve(
        query=query,
        k=3,
    )

    raptor_leaf = raptor_results.get(
        "leaf",
        [],
    )

    raptor_clusters = raptor_results.get(
        "clusters",
        [],
    )

    print(
        "RAPTOR leaf results:",
        len(raptor_leaf),
    )

    print(
        "RAPTOR cluster results:",
        len(raptor_clusters),
    )

In [129]:
# Self-RAG

self_rag_answer = ""

if decision == "NOT_ANSWERABLE":

    self_rag = SelfRAG(
        llm=llm,
        retriever=retriever,
    )

    self_rag_answer = self_rag.invoke(
        query,
        max_retries=2,
    )

    print("Self-RAG completed.")

In [130]:
# Long-context processing

long_context_answer = ""

if decision == "NOT_ANSWERABLE":

    long_context = LongContext(
        model="llama3:latest",
    )

    long_context_result = long_context.run(
        documents=reranked_documents,
        query=query,
        compress=True,
    )

    if isinstance(
        long_context_result,
        dict,
    ):
        long_context_answer = long_context_result.get(
            "answer",
            "",
        )
    else:
        long_context_answer = str(long_context_result)

In [131]:
# Build final context

context = ""

if decision == "NOT_ANSWERABLE":

    reranked_context = "\n\n".join(
        document.page_content for document in reranked_documents
    )

    if current_query:

        # Current/live queries:
        # Only use the freshness-aware web evidence.

        context = reranked_context

    else:

        # Stable external RAG:
        # Combine the advanced retrieval evidence.

        context_parts = [
            reranked_context,
        ]

        if raptor_leaf:
            context_parts.append(
                "RAPTOR Leaf Evidence:\n"
                + "\n\n".join(str(item) for item in raptor_leaf)
            )

        if raptor_clusters:
            context_parts.append(
                "RAPTOR Cluster Evidence:\n"
                + "\n\n".join(str(item) for item in raptor_clusters)
            )

        if self_rag_answer:
            context_parts.append("Self-RAG Evidence:\n" + self_rag_answer)

        if long_context_answer:
            context_parts.append("Long-Context Evidence:\n" + long_context_answer)

        context = "\n\n".join(context_parts)

    print(
        "Final context length:",
        len(context),
    )

In [132]:
# Final generation

if decision == "NOT_ANSWERABLE":

    answer = generation_chain.invoke(
        {
            "context": context,
            "question": query,
        }
    )

In [133]:
# Direct Ollama answer

if decision == "ANSWERABLE":

    response = llm.invoke(query)

    answer = response.content

In [134]:
# Store conversation memory

conversation_memory.add_message(HumanMessage(content=query))

conversation_memory.add_message(AIMessage(content=answer))

In [135]:
# Evaluation

if decision == "ANSWERABLE":

    answer_relevance = evaluator.evaluate_answer_relevance(
        question=query,
        answer=answer,
    )

    evaluation_results = {
        "answer_relevance": answer_relevance,
    }

else:

    context_relevance = evaluator.evaluate_context_relevance(
        question=query,
        context=context,
    )

    faithfulness = evaluator.evaluate_faithfulness(
        context=context,
        answer=answer,
    )

    answer_relevance = evaluator.evaluate_answer_relevance(
        question=query,
        answer=answer,
    )

    evaluation_results = {
        "context_relevance": context_relevance,
        "faithfulness": faithfulness,
        "answer_relevance": answer_relevance,
    }

print("Evaluation:")
for metric, value in evaluation_results.items():
    print(f"{metric}: {value}")

Evaluation:
answer_relevance: RELEVANT


In [136]:
# Final output

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)

print(answer)

print("=" * 60)

print(
    "\nDecision:",
    decision,
)

print(
    "Mode:",
    (
        "Direct Ollama"
        if decision == "ANSWERABLE"
        else ("Current Web RAG" if current_query else "Advanced RAG")
    ),
)


FINAL ANSWER
During the construction of the Saint Lawrence Seaway project in the late 1950s, several villages and communities were intentionally submerged or relocated to make way for the new waterway. In Ontario, Canada, the following specific villages were affected:

1. Rogers City: This village was located near the present-day town of Cornwall, Ontario. The village was intentionally flooded in 1958 to make way for the seaway.
2. Dickinson's Landing: This small village was located near the present-day town of Morrisburg, Ontario. The village was also intentionally flooded in 1958.
3. Dickinson's Landing was a small community that was home to about 100 people. The village was relocated to higher ground, and many of its residents were relocated to nearby Morrisburg.

These villages were submerged as part of the construction of the Saint Lawrence Seaway, which was a major engineering project that connected the Great Lakes to the Atlantic Ocean. The project was completed in 1959, and it